In [1]:
import cv2
import numpy as np
import time
import chess

In [2]:
def crop_by_percentage(image, percent):
        h, w = image.shape[:2]
        dx = int(w * percent)
        dy = int(h * percent)
        cropped = image[dy:h - dy, dx:w - dx]
        return cropped


In [9]:
def get_dots(image, DEBUG=False):
    # Define two HLS color ranges
    lower = np.array([115, 50, 70])
    upper = np.array([140, 230, 255])
    
    # Create masks
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HLS)
    colorMask = cv2.inRange(hsv, lower, upper)

    if DEBUG:
        cv2.imshow('Color Mask', colorMask)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()
    
    # Find contours in binary image
    contours, _ = cv2.findContours(colorMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Get center points of valid contours
    centers = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 10:
            M = cv2.moments(cnt)
            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                centers.append((cx, cy))

    return centers
    # Proceed if exactly 4 red dots found
    #if len(centers) != 4:
    #    raise Exception("not 4")

In [4]:
def divide_into_grid(image2, shave=0.1):
    h, w = image2.shape[:2]
    
    grid = []
    for y in range(8):
        for x in range(8):
            x1 = (x * w) // 8
            y1 = (y * h) // 8
            x2 = x1 + (w//8)
            y2 = y1 + (h//8)
            cell = image2[y1:y2, x1:x2]
            cell = crop_by_percentage(cell, shave)
            grid.append(cell)
    return grid



In [37]:
def get_occupancy(board):
    occupancy = np.zeros(64, dtype=np.bool)
    for i in range(64):
        occupancy[i] = (board.piece_at(i) != None)
    
    occupancy = occupancy.reshape((8,8))
    occupancy = np.flip(occupancy, axis=0)
    return occupancy

def get_maskmap(board):
    maskmap = {}
    maskmap[""] = get_occupancy(board)
    for move in board.legal_moves:
        board.push(move)
        maskmap[move.uci()] = get_occupancy(board)
        board.pop()
    return maskmap

In [39]:
prevCrop = None

DEBUG = True

playboard = chess.Board("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
camera = cv2.VideoCapture(0)

while True:
    time.sleep(1)
    # Load image and apply median blur
    #image = cv2.imread('image.jpg')
    #image = cv2.medianBlur(image, 5)
    _, image = camera.read()
    
    if DEBUG:
        cv2.imshow('Raw', image)
        cv2.waitKey(15000)
        cv2.destroyAllWindows()
    
    centers = get_dots(image, DEBUG)

    if len(centers) != 4:
        if prevCrop != None:
            centers = prevCrop
        else:
            print("Crop error", len(centers))
            continue

    prevCrop = centers
    
    pts = np.array(centers, dtype='float32')
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    ordered = np.array([
        pts[np.argmin(s)],       # top-left
        pts[np.argmin(diff)],    # top-right
        pts[np.argmax(s)],       # bottom-right
        pts[np.argmax(diff)]     # bottom-left
    ])
    
    width = int(max(np.linalg.norm(ordered[0] - ordered[1]), np.linalg.norm(ordered[2] - ordered[3])))
    height = int(max(np.linalg.norm(ordered[0] - ordered[3]), np.linalg.norm(ordered[1] - ordered[2])))
    dst_pts = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype='float32')
    
    matrix = cv2.getPerspectiveTransform(ordered, dst_pts)
    cropped = cv2.warpPerspective(image, matrix, (width, height))
    
    cropped = crop_by_percentage(cropped, 0.02)

    
    # Display the result
    if DEBUG:
        cv2.imshow('Cropped Area', cropped)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()

    canny = cv2.Canny(cropped, 120, 240)
    
    if DEBUG:
        cv2.imshow('Cropped Area', canny)
        cv2.waitKey(5000)
        cv2.destroyAllWindows()

    # Example usage
    grid_images = divide_into_grid(canny)
    
    arr = np.array(grid_images)
    arr = arr.reshape(8, 8, arr.shape[1] * arr.shape[2])
    arr = np.sum(arr, axis=2)
    #print(arr)
    
    #for i in range(64):
        # Show first cell as example
        #print(np.sum(grid_images[i]))
        #cv2.imshow('First Cell', grid_images[i])
        #cv2.waitKey(5000)
        #cv2.destroyAllWindows()

    playmasks = get_maskmap(playboard)

    bestscore = -99999999999
    bestmove = ""
    
    for key, value in playmasks.items():
        on_right = np.sum(arr[value]).astype(np.int64)
        off_wrong = np.sum(arr[~value]).astype(np.int64)
        score = on_right - off_wrong
        if score > bestscore:
            bestscore = score
            bestmove = key

    print(bestscore, bestmove)
    
    if DEBUG:
        break
        

931515 e2e4
